In [0]:
from pyspark.sql import functions as F

landing_path = "/Volumes/cloud_pricing/raw/landing/azure_retail"
raw_table = "cloud_pricing.bronze.retail_prices_raw"
audit_table = "cloud_pricing.bronze.load_audit"
regions = ["centralindia", "eastus"]  # same list as the ADF pipeline param

# not parsing items here, just keeping them as json strings. silver does the parsing
page_schema = "Items ARRAY<STRING>, NextPageLink STRING, Count INT"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {raw_table} (
  run_id STRING, run_date DATE, region STRING, collector STRING,
  source_file STRING, item STRING, ingested_at TIMESTAMP
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
  run_id STRING, run_date DATE, region STRING, pages INT, items BIGINT,
  status STRING, problems STRING, checked_at TIMESTAMP
)""")

In [0]:
# a failed ADF run is never completed later, a rerun gets a new run_id. so anything already checked is skipped
loaded = [r.run_id for r in spark.sql(f"select distinct run_id from {audit_table}").collect()]

new_runs = []
for d in dbutils.fs.ls(landing_path):
    run_date = d.name.strip("/").split("=")[1]
    for r in dbutils.fs.ls(d.path):
        run_id = r.name.strip("/").split("=")[1]
        if run_id not in loaded:
            new_runs.append((run_date, run_id, r.path))

print(len(new_runs), "new runs")
for x in new_runs:
    print(x[1])

In [0]:
def check_run(pages):
    stats = pages.groupBy("region").agg(
        F.count("*").alias("pages"),
        F.sum(F.size("page.Items")).alias("items"),
        F.sum("page.Count").alias("items_reported"),
        F.sum(F.when(F.col("page.NextPageLink").isNull(), 1).otherwise(0)).alias("last_pages"),
        F.sum(F.when(F.col("page.Items").isNull(), 1).otherwise(0)).alias("bad_pages")
    ).collect()
    by_region = {s["region"]: s for s in stats}

    problems = {}
    for region in regions:
        s = by_region.get(region)
        if s is None:
            problems[region] = "no files"
            continue
        issues = []
        if s["bad_pages"] > 0:
            issues.append(f"{s['bad_pages']} unreadable pages")
        # only the final page has NextPageLink = null. 0 means paging stopped midway
        if s["last_pages"] != 1:
            issues.append(f"{s['last_pages']} last pages, expected 1")
        if s["items"] != s["items_reported"]:
            issues.append(f"{s['items']} items but Count says {s['items_reported']}")
        if issues:
            problems[region] = "; ".join(issues)

    return by_region, problems

In [0]:
for run_date, run_id, path in new_runs:
    # works for both layouts - laptop run has page_0001.json.. files, ADF has one prices.json per region with a page per line.
    # _manifest.json is at run root so */*.json doesn't pick it up
    pages = (spark.read.text(f"{path}/*/*.json")
             .select(F.col("_metadata.file_path").alias("source_file"), "value")
             .withColumn("region", F.regexp_extract("source_file", r"run_id=[^/]+/([^/]+)/", 1))
             .withColumn("page", F.from_json("value", page_schema)))

    by_region, problems = check_run(pages)
    status = "incomplete" if problems else "loaded"

    if status == "loaded":
        items = (pages.select("region", "source_file", F.explode("page.Items").alias("item"))
                 .withColumn("run_id", F.lit(run_id))
                 .withColumn("run_date", F.to_date(F.lit(run_date)))
                 .withColumn("collector", F.when(F.col("source_file").endswith("prices.json"), "adf").otherwise("local_python"))
                 .withColumn("ingested_at", F.current_timestamp()))

        # replaceWhere so a rerun of the same run_id overwrites instead of duplicating
        items.select("run_id", "run_date", "region", "collector", "source_file", "item", "ingested_at") \
            .write.mode("overwrite").option("replaceWhere", f"run_id = '{run_id}'").saveAsTable(raw_table)

    # audit row goes in either way, so failed runs are visible too
    rows = []
    for region in regions:
        s = by_region.get(region)
        rows.append((run_id, run_date, region, s["pages"] if s else 0, s["items"] if s else 0, status, problems.get(region)))

    audit_df = spark.createDataFrame(rows, "run_id STRING, run_date STRING, region STRING, pages INT, items BIGINT, status STRING, problems STRING")
    audit_df.withColumn("run_date", F.to_date("run_date")) \
        .withColumn("checked_at", F.current_timestamp()) \
        .write.mode("append").saveAsTable(audit_table)

    print(run_id, status, problems if problems else "")

In [0]:
display(spark.sql(f"select run_id, region, status, pages, items, problems from {audit_table} order by checked_at desc, region"))

In [0]:
# quick eyeball of what landed
display(spark.table(raw_table).groupBy("run_id", "collector").count().orderBy("run_id"))